# Student Enrollment Forecasting

## Project Overview

Forecasts **daily student enrollment counts** over a 14-day horizon. Synthetic data spans 2 years with strong annual cycles (academic calendar), registration surges, and gradual growth.

**Models**: Naive, LazyPredict, FLAML, StatsForecast. **Horizon**: 14 days.

## Learning Objectives

1. Understand time-series patterns (trend, seasonality, noise)
2. Build naive and seasonal-naive baselines
3. Engineer lag and rolling features for tabular ML
4. Use LazyPredict for quick regression benchmarking
5. Apply FLAML AutoML (excluding XGBoost due to compatibility)
6. Use StatsForecast (AutoARIMA, AutoETS, SeasonalNaive)
7. Compare all approaches with MAE / RMSE / MAPE

## Problem Statement

Given historical daily enrollment data, predict the next 14 days. Enrollment forecasts support class scheduling, faculty hiring, budgeting, and facilities planning for educational institutions.

## Why This Project Matters

Universities and schools must plan resources months in advance. Enrollment determines tuition revenue, classroom needs, housing demand, and staffing levels. Overestimating wastes budget; underestimating creates overcrowded classes and poor student experience. Accurate forecasts enable optimal resource allocation.

## Dataset Overview

Synthetic dataset:
- 730 days (2 years) of daily enrollment activity
- Strong academic calendar seasonality (fall/spring registration peaks)
- Weekly pattern (lower on weekends)
- Registration deadline surges
- Gradual growth trend (institutional growth)

| Column | Description |
|--------|-------------|
| `date` | Date |
| `enrollments` | Daily student enrollment count |

## Dataset Source and License Notes

Synthetically generated for educational purposes. No external download required.

## Environment Setup

In [1]:
import subprocess, importlib
for pkg in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'statsforecast', 'flaml', 'lazypredict']:
    try:
        importlib.import_module(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call(['pip', 'install', '-q', pkg])
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Imports complete.")

Imports complete.


## Configuration / Constants

In [3]:
SEED = 42
N_POINTS = 730
HORIZON = 14
VAL_SIZE = 14
TEST_SIZE = 14
SEASON_LENGTH = 7
FREQ = 'D'
TARGET = 'enrollments'
np.random.seed(SEED)
print(f"Config: {N_POINTS} points, horizon={HORIZON}, season={SEASON_LENGTH}")

Config: 730 points, horizon=14, season=7


## Dataset Generation

In [4]:
dates = pd.date_range(start='2022-01-01', periods=N_POINTS, freq='D')
t = np.arange(N_POINTS)

# Base with growth
base = 50 + 0.015 * t

# Academic calendar: peaks in Jan (spring) and Aug/Sep (fall)
seasonal = (25 * np.exp(-((t % 365 - 15) ** 2) / 200) +  # January spring registration
            35 * np.exp(-((t % 365 - 240) ** 2) / 200) +  # September fall registration
            10 * np.exp(-((t % 365 - 135) ** 2) / 300))    # May summer session

# Weekly pattern
weekly = np.zeros(N_POINTS)
for i in range(N_POINTS):
    dow = dates[i].dayofweek
    if dow <= 4:
        weekly[i] = 8
    elif dow == 5:
        weekly[i] = -10
    else:
        weekly[i] = -20

# Registration deadline surges
for y in range(2):
    for deadline_doy in [10, 130, 235]:  # Jan, May, Aug deadlines
        d = y * 365 + deadline_doy
        if d < N_POINTS:
            for j in range(min(3, N_POINTS - d)):
                seasonal[d + j] += 20 * np.exp(-0.5 * j)

noise = np.random.normal(0, 4, N_POINTS)
enrollments = base + seasonal + weekly + noise
enrollments = np.maximum(enrollments, 5).round(0).astype(int)

df = pd.DataFrame({'date': dates, 'enrollments': enrollments})
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (730, 2)


,date,enrollments
0,2022-01-01,50
1,2022-01-02,39
2,2022-01-03,71
3,2022-01-04,76
4,2022-01-05,71


## Data Validation Checks

In [5]:
assert df.isnull().sum().sum() == 0, "Missing values"
assert (df[TARGET] > 0).all(), "Non-positive values"
assert df['date'].is_monotonic_increasing, "Not sorted"
assert len(df) == N_POINTS, "Row count mismatch"
print("All validation checks passed.")

All validation checks passed.


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(df['date'], df[TARGET], linewidth=0.6)
axes[0, 0].set_title('enrollments Over Time')
axes[0, 1].hist(df[TARGET], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution')
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(df[TARGET], ax=axes[1, 0])
axes[1, 0].set_xlim(0, min(60, N_POINTS // 2))
axes[1, 0].set_title('Autocorrelation')
rw = min(SEASON_LENGTH * 2, N_POINTS // 4)
axes[1, 1].plot(df['date'], df[TARGET].rolling(rw).mean())
axes[1, 1].set_title(f'Rolling {rw}-period Mean')
plt.tight_layout()
plt.savefig('eda.png', dpi=100, bbox_inches='tight')
plt.show()
print("EDA complete.")

EDA complete.


## Target Analysis

In [7]:
print("enrollments Statistics:")
print(df[TARGET].describe())
print(f"\nCV: {df[TARGET].std() / df[TARGET].mean():.3f}")
print(f"Skewness: {df[TARGET].skew():.3f}")

enrollments Statistics:
count    730.000000
mean      61.963014
std       14.931669
min       28.000000
25%       54.000000
50%       63.000000
75%       70.000000
max      121.000000
Name: enrollments, dtype: float64

CV: 0.241
Skewness: 0.172


## Train / Validation / Test Split Strategy

Time-aware split (no shuffling):
- **Train**: all except last 28
- **Validation**: 14 points
- **Test**: last 14 points

In [8]:
train = df.iloc[:-(VAL_SIZE + TEST_SIZE)].copy()
val = df.iloc[-(VAL_SIZE + TEST_SIZE):-TEST_SIZE].copy()
test = df.iloc[-TEST_SIZE:].copy()
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 702, Val: 14, Test: 14


## Preprocessing Strategy

Minimal preprocessing — tree models handle raw features. Features created next.

In [9]:
df_full = df.copy().sort_values('date').reset_index(drop=True)
print('Data ready.')

Data ready.


## Feature Engineering

In [10]:
def create_features(data):
    d = data.copy()
    d['dow'] = d['date'].dt.dayofweek
    d['month'] = d['date'].dt.month
    d['day'] = d['date'].dt.day
    d['week_of_year'] = d['date'].dt.isocalendar().week.astype(int)
    for lag in [1, 7, 14, 28]:
        d[f'lag_{lag}'] = d[TARGET].shift(lag)
    for w in [7, 14, 28]:
        d[f'rmean_{w}'] = d[TARGET].shift(1).rolling(w).mean()
        d[f'rstd_{w}'] = d[TARGET].shift(1).rolling(w).std()
    return d

df_feat = create_features(df_full).dropna().reset_index(drop=True)
feature_cols = [c for c in df_feat.columns if c not in ['date', TARGET]]
print(f"Features: {len(feature_cols)} columns, {len(df_feat)} rows")

Features: 14 columns, 702 rows


## Baseline Model — Naive Forecasts

In [11]:
def calc_metrics(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100
    print(f"{name:30s} | MAE: {mae:10.1f} | RMSE: {rmse:10.1f} | MAPE: {mape:5.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []
naive_pred = np.full(TEST_SIZE, train[TARGET].iloc[-1])
results.append(calc_metrics(test[TARGET].values, naive_pred, "Naive (Last Value)"))

src = df_full[TARGET].values
ti = len(df_full) - TEST_SIZE
sn_pred = src[ti - SEASON_LENGTH:ti - SEASON_LENGTH + TEST_SIZE]
results.append(calc_metrics(test[TARGET].values, sn_pred, f"Seasonal Naive ({SEASON_LENGTH})"))

Naive (Last Value)             | MAE:       18.9 | RMSE:       20.9 | MAPE: 29.13%
Seasonal Naive (7)             | MAE:        2.8 | RMSE:        4.1 | MAPE:  5.91%


## LazyPredict Benchmark (Lag-Feature Tabular)

In [12]:
from lazypredict.Supervised import LazyRegressor

ct = len(df_feat) - TEST_SIZE
cv = ct - VAL_SIZE
X_tr = df_feat.iloc[:cv][feature_cols]
y_tr = df_feat.iloc[:cv][TARGET]
X_va = df_feat.iloc[cv:ct][feature_cols]
y_va = df_feat.iloc[cv:ct][TARGET]

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None, predictions=True)
lp_models, lp_preds = reg.fit(X_tr, X_va, y_tr, y_va)
print("\nLazyPredict Top 10:")
print(lp_models.head(10).to_string())


LazyPredict Top 10:
                             Adjusted R-Squared  R-Squared       RMSE  Time Taken
Model                                                                            
KernelRidge                          405.592294 -30.122484  62.611203    0.060450
GaussianProcessRegressor              84.243874  -5.403375  28.400068    0.069780
QuantileRegressor                     14.254861  -0.019605  11.332633    0.061854
DummyRegressor                        14.003501  -0.000269  11.224665    0.009213
ExtraTreeRegressor                    11.851588   0.165262  10.253919    0.010711
MLPRegressor                          10.248288   0.288593   9.466161    0.448677
OrthogonalMatchingPursuitCV            6.477014   0.578691   7.284758    0.027625
DecisionTreeRegressor                  6.388934   0.585467   7.225945    0.013215
OrthogonalMatchingPursuit              6.250615   0.596107   7.132608    0.015053
Lasso                                  6.008524   0.614729   6.966235    0.01

## FLAML AutoML Run

In [13]:
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train=X_tr, y_train=y_tr,
    task='regression', time_budget=30, metric='mae',
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
print(f"FLAML best: {automl.best_estimator}")
flaml_val = automl.predict(X_va)
results.append(calc_metrics(y_va.values, flaml_val, f"FLAML ({automl.best_estimator})"))

X_te = df_feat.iloc[ct:][feature_cols]
y_te = df_feat.iloc[ct:][TARGET]
flaml_test = automl.predict(X_te)
results.append(calc_metrics(y_te.values, flaml_test, f"FLAML Test ({automl.best_estimator})"))

FLAML best: lgbm
FLAML (lgbm)                   | MAE:        4.9 | RMSE:        6.1 | MAPE:  9.32%
FLAML Test (lgbm)              | MAE:        2.9 | RMSE:        3.7 | MAPE:  5.49%


## StatsForecast — Statistical Forecasting

In [14]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive as SFSeasonalNaive

sf_df = df_full[['date', TARGET]].copy()
sf_df.columns = ['ds', 'y']
sf_df['unique_id'] = 'series1'
sf_df = sf_df[['unique_id', 'ds', 'y']]

sf_train = sf_df.iloc[:-TEST_SIZE]
sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LENGTH), AutoETS(season_length=SEASON_LENGTH),
            SFSeasonalNaive(season_length=SEASON_LENGTH)],
    freq=FREQ, n_jobs=1
)
sf.fit(sf_train)
sf_preds = sf.predict(h=TEST_SIZE).reset_index()

y_test_sf = test[TARGET].values
for col in ['AutoARIMA', 'AutoETS', 'SeasonalNaive']:
    if col in sf_preds.columns:
        results.append(calc_metrics(y_test_sf, sf_preds[col].values, f"SF {col}"))
print("StatsForecast complete.")

SF AutoARIMA                   | MAE:        3.0 | RMSE:        4.1 | MAPE:  6.32%
SF AutoETS                     | MAE:        2.7 | RMSE:        3.7 | MAPE:  5.76%
SF SeasonalNaive               | MAE:        2.9 | RMSE:        4.3 | MAPE:  6.33%
StatsForecast complete.


## Top 3 Model Selection

In [15]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
print("All Results:")
print(results_df.to_string(index=False))
print("\nTop 3:")
print(results_df.head(3).to_string(index=False))

All Results:
             model       MAE      RMSE      MAPE
        SF AutoETS  2.716494  3.728589  5.757622
Seasonal Naive (7)  2.785714  4.097037  5.909422
 FLAML Test (lgbm)  2.873161  3.726766  5.494981
  SF SeasonalNaive  2.928571  4.284524  6.325981
      SF AutoARIMA  2.998695  4.120942  6.315001
      FLAML (lgbm)  4.867215  6.079267  9.320920
Naive (Last Value) 18.928571 20.851516 29.132937

Top 3:
             model      MAE     RMSE     MAPE
        SF AutoETS 2.716494 3.728589 5.757622
Seasonal Naive (7) 2.785714 4.097037 5.909422
 FLAML Test (lgbm) 2.873161 3.726766 5.494981


## Final Training and Evaluation of Top 3

In [16]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test['date'].values, test[TARGET].values, 'ko-', label='Actual', ms=4)
ax.plot(test['date'].values, flaml_test, 's--', label=f'FLAML ({automl.best_estimator})', ms=4)
for col in ['AutoARIMA', 'AutoETS']:
    if col in sf_preds.columns:
        ax.plot(test['date'].values[:len(sf_preds)], sf_preds[col].values, 'o--', label=f'SF {col}', ms=4)
ax.set_title('Forecast vs Actual — Test Set')
ax.legend()
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Error Analysis

In [17]:
errors = test[TARGET].values - flaml_test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(errors)), errors, alpha=0.7)
axes[0].set_title('Residuals (Actual - FLAML)')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[1].hist(errors, bins=10, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Mean residual: {np.mean(errors):.2f}, Std: {np.std(errors):.2f}")

Mean residual: 0.12, Std: 3.72


## Interpretation and Business Insight

- Enrollment follows a strong academic calendar cycle
- Fall registration (Aug/Sep) is the largest peak
- Spring registration (Jan) is the second peak
- Registration deadlines create sharp short-term surges
- Weekend enrollment activity drops significantly

## Limitations

1. Synthetic — real enrollment depends on demographics, admissions, economy
2. No distinction between new and returning students
3. No program-level breakdown
4. No demographic or economic features
5. No admissions funnel data (applications → admits → enrollments)

## How to Improve This Project

1. Add demographic data (high school graduates, population trends)
2. Include economic indicators (unemployment, tuition costs)
3. Model by program/department for detailed planning
4. Add admissions funnel conversion rates
5. Use cohort-based models for retention forecasting

## Production Considerations

- Enrollment forecasting for budget planning
- Class section and faculty allocation
- Housing and facilities demand estimation
- Financial aid disbursement planning

## Common Mistakes

1. Ignoring the academic calendar in model features
2. Not distinguishing registration surges from steady-state enrollment
3. Using annual averages when semester-level data is needed
4. Ignoring retention rates and dropout patterns
5. Not accounting for demographic trends in long-term forecasts

## Mini Challenge / Exercises

1. Identify the registration peaks from the data
2. Build separate models for fall vs spring enrollment
3. Estimate year-over-year growth rate
4. Create a registration deadline detector from daily changes

## Final Summary / Key Takeaways

1. Student enrollment has the strongest calendar-driven seasonality
2. Fall registration is the dominant annual event
3. Registration deadlines create predictable short-term surges
4. Real deployment needs demographic and admissions funnel data
5. Budget planning requires semester-level aggregation, not daily

In [18]:
import json
metrics = {
    'project': 'Student Enrollment Forecasting',
    'best_model': results_df.iloc[0]['model'],
    'best_MAE': float(results_df.iloc[0]['MAE']),
    'best_RMSE': float(results_df.iloc[0]['RMSE']),
    'best_MAPE': float(results_df.iloc[0]['MAPE']),
    'all_results': results_df.to_dict('records')
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved.")
print("\n=== Student Enrollment Forecasting — Complete ===")

Metrics saved.

=== Student Enrollment Forecasting — Complete ===
